In [ ]:
!pip install flask requests


In [ ]:
import hashlib
import json
import uuid
import requests
import threading
import time

from datetime import datetime
from flask import Flask, jsonify, request
from urllib.parse import urlparse

Create the Blockchain class

In [ ]:
class Blockchain:

    def __init__(self, node_address):
        self.chain = []
        self.transactions = []
        self.nodes = set()
        self.node_address = node_address

        # Create Genesis Block
        self.create_block(proof=1, previous_hash='0')


    # ------------------------------------------------
    # CREATE BLOCK
    # ------------------------------------------------

    def create_block(self, proof, previous_hash):

        block = {
            'index': len(self.chain) + 1,
            'timestamp': str(datetime.now()),
            'proof': proof,
            'previous_hash': previous_hash,
            'transactions': self.transactions.copy()
        }

        self.transactions = []

        self.chain.append(block)

        return block


    # ------------------------------------------------
    # GET PREVIOUS BLOCK
    # ------------------------------------------------

    def get_previous_block(self):

        return self.chain[-1]


    # ------------------------------------------------
    # PROOF OF WORK
    # ------------------------------------------------

    def proof_of_work(self, previous_proof):

        new_proof = 1

        check_proof = False

        while check_proof is False:

            hash_operation = hashlib.sha256(
                str(new_proof**2 - previous_proof**2).encode()
            ).hexdigest()

            if hash_operation[:4] == '0000':
                check_proof = True
            else:
                new_proof += 1

        return new_proof


    # ------------------------------------------------
    # HASH BLOCK
    # ------------------------------------------------

    def hash(self, block):

        encoded_block = json.dumps(
            block,
            sort_keys=True
        ).encode()

        return hashlib.sha256(encoded_block).hexdigest()


    # ------------------------------------------------
    # CHECK BLOCKCHAIN VALIDITY
    # ------------------------------------------------

    def is_chain_valid(self, chain):

        previous_block = chain[0]

        block_index = 1

        while block_index < len(chain):

            block = chain[block_index]

            # Check previous hash
            if block['previous_hash'] != self.hash(previous_block):
                return False

            previous_proof = previous_block['proof']
            current_proof = block['proof']

            hash_operation = hashlib.sha256(
                str(
                    current_proof**2 - previous_proof**2
                ).encode()
            ).hexdigest()

            if hash_operation[:4] != '0000':
                return False

            previous_block = block
            block_index += 1

        return True


    # ------------------------------------------------
    # ADD TRANSACTION
    # ------------------------------------------------

    def add_transaction(self, sender, receiver, amount):

        self.transactions.append({
            'sender': sender,
            'receiver': receiver,
            'amount': amount
        })

        previous_block = self.get_previous_block()

        return previous_block['index'] + 1


    # ------------------------------------------------
    # ADD NODE
    # ------------------------------------------------

    def add_node(self, address):

        parsed_url = urlparse(address)

        if parsed_url.netloc:
            self.nodes.add(parsed_url.netloc)

        elif parsed_url.path:
            self.nodes.add(parsed_url.path)


    # ------------------------------------------------
    # REPLACE CHAIN
    # ------------------------------------------------

    def replace_chain(self):

        network = self.nodes

        longest_chain = None
        max_length = len(self.chain)

        for node in network:

            try:

                response = requests.get(
                    f'http://{node}/get_chain'
                )

                if response.status_code == 200:

                    length = response.json()['length']
                    chain = response.json()['chain']

                    if (
                        length > max_length
                        and self.is_chain_valid(chain)
                    ):

                        max_length = length
                        longest_chain = chain

            except Exception as e:
                print(f"Could not connect to {node}")

        if longest_chain:

            self.chain = longest_chain

            return True

        return False

Create the 3 blockchain nodes

In [ ]:
blockchains = {
    5001: Blockchain(5001),
    5002: Blockchain(5002),
    5003: Blockchain(5003)
}

In [ ]:
for port, blockchain in blockchains.items():
    print(
        f"Node {port} -> "
        f"Blockchain length: {len(blockchain.chain)}"
    )

Node 5001 -> Blockchain length: 1
Node 5002 -> Blockchain length: 1
Node 5003 -> Blockchain length: 1


Create Flask APIs

In [ ]:
def create_app(port):

    app = Flask(__name__)

    blockchain = blockchains[port]


    # ================================================
    # CONNECT NODE
    # ================================================

    @app.route('/connect_node', methods=['POST'])
    def connect_node():

        json_data = request.get_json()

        nodes = json_data.get('nodes')

        if not nodes:
            return jsonify({
                'message': 'No nodes provided'
            }), 400

        for node in nodes:
            blockchain.add_node(node)

        return jsonify({
            'message': 'All the nodes are now connected.',
            'total_nodes': list(blockchain.nodes)
        }), 201


    # ================================================
    # ADD TRANSACTION
    # ================================================

    @app.route('/add_transaction', methods=['POST'])
    def add_transaction():

        json_data = request.get_json()

        sender = json_data['sender']
        receiver = json_data['receiver']
        amount = json_data['amount']

        index = blockchain.add_transaction(
            sender,
            receiver,
            amount
        )

        return jsonify({
            'message': f'Transaction will be added to Block {index}'
        }), 201


    # ================================================
    # MINE BLOCK
    # ================================================

    @app.route('/mine_block', methods=['GET'])
    def mine_block():

        previous_block = blockchain.get_previous_block()

        previous_proof = previous_block['proof']

        proof = blockchain.proof_of_work(previous_proof)

        previous_hash = blockchain.hash(previous_block)


        # Mining reward
        blockchain.transactions.append({
            'sender': str(uuid.uuid4()),
            'receiver': f'Miner_{port}',
            'amount': 1
        })


        block = blockchain.create_block(
            proof,
            previous_hash
        )

        return jsonify({
            'message': 'Congratulations, you just mined a block!',
            'index': block['index'],
            'timestamp': block['timestamp'],
            'proof': block['proof'],
            'previous_hash': block['previous_hash'],
            'transactions': block['transactions']
        }), 200


    # ================================================
    # GET CHAIN
    # ================================================

    @app.route('/get_chain', methods=['GET'])
    def get_chain():

        return jsonify({
            'chain': blockchain.chain,
            'length': len(blockchain.chain)
        }), 200


    # ================================================
    # REPLACE CHAIN
    # ================================================

    @app.route('/replace_chain', methods=['GET'])
    def replace_chain():

        is_chain_replaced = blockchain.replace_chain()

        if is_chain_replaced:

            return jsonify({
                'message': 'The chain was replaced by the longest chain.',
                'new_chain': blockchain.chain
            }), 200

        else:

            return jsonify({
                'message': 'The current chain is already the longest chain.',
                'actual_chain': blockchain.chain
            }), 200


    return app

Start all 3 nodes

In [ ]:
apps = {}

for port in [5001, 5002, 5003]:

    app = create_app(port)

    apps[port] = app

    thread = threading.Thread(
        target=app.run,
        kwargs={
            'host': '0.0.0.0',
            'port': port,
            'debug': False,
            'use_reloader': False
        }
    )

    thread.daemon = True
    thread.start()

    time.sleep(1)

print("All 3 blockchain nodes started!")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5002
 * Running on http://172.28.0.12:5002
INFO:werkzeug:Press CTRL+C to quit


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5003
 * Running on http://172.28.0.12:5003
INFO:werkzeug:Press CTRL+C to quit


All 3 blockchain nodes started!


Test Node 5001

In [ ]:
response = requests.get(
    'http://127.0.0.1:5001/get_chain'
)

print(response.json())

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:49:24] "GET /get_chain HTTP/1.1" 200 -


{'chain': [{'index': 1, 'previous_hash': '0', 'proof': 1, 'timestamp': '2026-08-27 04:45:32.109740', 'transactions': []}], 'length': 1}


Connect the three nodes

Connect 5001

In [ ]:
requests.post(
    'http://127.0.0.1:5001/connect_node',
    json={
        'nodes': [
            'http://127.0.0.1:5002',
            'http://127.0.0.1:5003'
        ]
    }
).json()

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:49:28] "POST /connect_node HTTP/1.1" 201 -


{'message': 'All the nodes are now connected.',
 'total_nodes': ['127.0.0.1:5003', '127.0.0.1:5002']}

Connect 5002

In [ ]:
requests.post(
    'http://127.0.0.1:5002/connect_node',
    json={
        'nodes': [
            'http://127.0.0.1:5001',
            'http://127.0.0.1:5003'
        ]
    }
).json()

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:49:33] "POST /connect_node HTTP/1.1" 201 -


{'message': 'All the nodes are now connected.',
 'total_nodes': ['127.0.0.1:5001', '127.0.0.1:5003']}

Connect 5003

In [ ]:
requests.post(
    'http://127.0.0.1:5003/connect_node',
    json={
        'nodes': [
            'http://127.0.0.1:5001',
            'http://127.0.0.1:5002'
        ]
    }
).json()

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:49:35] "POST /connect_node HTTP/1.1" 201 -


{'message': 'All the nodes are now connected.',
 'total_nodes': ['127.0.0.1:5001', '127.0.0.1:5002']}

Verify connections

In [ ]:
for port in [5001, 5002, 5003]:

    print(f"\nNODE {port}")

    print(
        blockchains[port].nodes
    )


NODE 5001
{'127.0.0.1:5003', '127.0.0.1:5002'}

NODE 5002
{'127.0.0.1:5001', '127.0.0.1:5003'}

NODE 5003
{'127.0.0.1:5001', '127.0.0.1:5002'}


Add transactions

In [ ]:
{
    "sender": "Nikhil",
    "receiver": "Aayush",
    "amount": 5
}

{'sender': 'Nikhil', 'receiver': 'Aayush', 'amount': 5}

Transaction 1

In [ ]:
requests.post(
    'http://127.0.0.1:5001/add_transaction',
    json={
        'sender': 'Aayush',
        'receiver': 'Nikhil Kale',
        'amount': 5
    }
).json()

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:56:38] "POST /add_transaction HTTP/1.1" 201 -


{'message': 'Transaction will be added to Block 2'}

Transaction 2

In [ ]:
requests.post(
    'http://127.0.0.1:5001/add_transaction',
    json={
        'sender': 'Nikhil',
        'receiver': 'Aayush',
        'amount': 10
    }
).json()

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:56:35] "POST /add_transaction HTTP/1.1" 201 -


{'message': 'Transaction will be added to Block 2'}

Transaction 3

In [ ]:
requests.post(
    'http://127.0.0.1:5001/add_transaction',
    json={
        'sender': 'Nikhil',
        'receiver': 'Aayush2',
        'amount': 15
    }
).json()

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:56:51] "POST /add_transaction HTTP/1.1" 201 -


{'message': 'Transaction will be added to Block 2'}

Check pending transactions

In [ ]:
print(blockchains[5001].transactions)

[{'sender': 'Nikhil', 'receiver': 'Aayush', 'amount': 10}, {'sender': 'Aayush', 'receiver': 'Nikhil Kale', 'amount': 5}, {'sender': 'Nikhil', 'receiver': 'Aayush2', 'amount': 15}]


Check chain lengths BEFORE mining

In [ ]:
for port in [5001, 5002, 5003]:

    response = requests.get(
        f'http://127.0.0.1:{port}/get_chain'
    )

    print(
        f"Node {port}: "
        f"{response.json()['length']} blocks"
    )

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:24] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:24] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:24] "GET /get_chain HTTP/1.1" 200 -


Node 5001: 1 blocks
Node 5002: 1 blocks
Node 5003: 1 blocks


Mine a block on Node 5001

In [ ]:
response = requests.get(
    'http://127.0.0.1:5001/mine_block'
)

result = response.json()

print(json.dumps(result, indent=4))

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:26] "GET /mine_block HTTP/1.1" 200 -


{
    "index": 2,
    "message": "Congratulations, you just mined a block!",
    "previous_hash": "a96b7d9fc9c397055435857d5b6a55288800b22e8da695730f320b7cd9a98610",
    "proof": 533,
    "timestamp": "2026-08-27 04:58:26.875281",
    "transactions": [
        {
            "amount": 10,
            "receiver": "Aayush",
            "sender": "Nikhil"
        },
        {
            "amount": 5,
            "receiver": "Nikhil Kale",
            "sender": "Aayush"
        },
        {
            "amount": 15,
            "receiver": "Aayush2",
            "sender": "Nikhil"
        },
        {
            "amount": 1,
            "receiver": "Miner_5001",
            "sender": "d54aab39-7b10-47f4-b51e-dc1785e8669e"
        }
    ]
}


Check all three chain lengths

In [ ]:
for port in [5001, 5002, 5003]:

    response = requests.get(
        f'http://127.0.0.1:{port}/get_chain'
    )

    print(
        f"Node {port}: "
        f"{response.json()['length']} blocks"
    )

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:33] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:33] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:33] "GET /get_chain HTTP/1.1" 200 -


Node 5001: 2 blocks
Node 5002: 1 blocks
Node 5003: 1 blocks


Demonstrate replace_chain()

Now Node 5002 knows about Node 5001 but doesn't have its longer blockchain.

In [ ]:
response = requests.get(
    'http://127.0.0.1:5002/replace_chain'
)

print(json.dumps(response.json(), indent=4))

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:35] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:35] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:35] "GET /replace_chain HTTP/1.1" 200 -


{
    "message": "The chain was replaced by the longest chain.",
    "new_chain": [
        {
            "index": 1,
            "previous_hash": "0",
            "proof": 1,
            "timestamp": "2026-08-27 04:45:32.109740",
            "transactions": []
        },
        {
            "index": 2,
            "previous_hash": "a96b7d9fc9c397055435857d5b6a55288800b22e8da695730f320b7cd9a98610",
            "proof": 533,
            "timestamp": "2026-08-27 04:58:26.875281",
            "transactions": [
                {
                    "amount": 10,
                    "receiver": "Aayush",
                    "sender": "Nikhil"
                },
                {
                    "amount": 5,
                    "receiver": "Nikhil Kale",
                    "sender": "Aayush"
                },
                {
                    "amount": 15,
                    "receiver": "Aayush2",
                    "sender": "Nikhil"
                },
                {
      

Check Node 5002

In [ ]:
response = requests.get(
    'http://127.0.0.1:5002/get_chain'
)

print(
    "Node 5002 chain length:",
    response.json()['length']
)

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:49] "GET /get_chain HTTP/1.1" 200 -


Node 5002 chain length: 2


Replace chain on Node 5003

In [ ]:
response = requests.get(
    'http://127.0.0.1:5003/replace_chain'
)

print(json.dumps(response.json(), indent=4))

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:50] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:50] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:50] "GET /replace_chain HTTP/1.1" 200 -


{
    "message": "The chain was replaced by the longest chain.",
    "new_chain": [
        {
            "index": 1,
            "previous_hash": "0",
            "proof": 1,
            "timestamp": "2026-08-27 04:45:32.109740",
            "transactions": []
        },
        {
            "index": 2,
            "previous_hash": "a96b7d9fc9c397055435857d5b6a55288800b22e8da695730f320b7cd9a98610",
            "proof": 533,
            "timestamp": "2026-08-27 04:58:26.875281",
            "transactions": [
                {
                    "amount": 10,
                    "receiver": "Aayush",
                    "sender": "Nikhil"
                },
                {
                    "amount": 5,
                    "receiver": "Nikhil Kale",
                    "sender": "Aayush"
                },
                {
                    "amount": 15,
                    "receiver": "Aayush2",
                    "sender": "Nikhil"
                },
                {
      

Final verification

In [ ]:
for port in [5001, 5002, 5003]:

    response = requests.get(
        f'http://127.0.0.1:{port}/get_chain'
    )

    data = response.json()

    print(
        f"Node {port} → "
        f"{data['length']} blocks"
    )

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:53] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:53] "GET /get_chain HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:58:53] "GET /get_chain HTTP/1.1" 200 -


Node 5001 → 2 blocks
Node 5002 → 2 blocks
Node 5003 → 2 blocks


Modify the code such that transactions are removed after they are added to the block.

In [ ]:
def create_block(self, proof, previous_hash):

    block = {
        'index': len(self.chain) + 1,
        'timestamp': str(datetime.now()),
        'proof': proof,
        'previous_hash': previous_hash,
        'transactions': self.transactions.copy()
    }

    self.transactions = []

    self.chain.append(block)

    return block

transactions were removed

In [ ]:
print(
    "Pending transactions on Node 5001:"
)

print(
    blockchains[5001].transactions
)

Pending transactions on Node 5001:
[]


Check the complete chain

In [ ]:
response = requests.get(
    'http://127.0.0.1:5001/get_chain'
)

data = response.json()

print(json.dumps(data, indent=4))

INFO:werkzeug:127.0.0.1 - - [27/Aug/2026 04:59:10] "GET /get_chain HTTP/1.1" 200 -


{
    "chain": [
        {
            "index": 1,
            "previous_hash": "0",
            "proof": 1,
            "timestamp": "2026-08-27 04:45:32.109740",
            "transactions": []
        },
        {
            "index": 2,
            "previous_hash": "a96b7d9fc9c397055435857d5b6a55288800b22e8da695730f320b7cd9a98610",
            "proof": 533,
            "timestamp": "2026-08-27 04:58:26.875281",
            "transactions": [
                {
                    "amount": 10,
                    "receiver": "Aayush",
                    "sender": "Nikhil"
                },
                {
                    "amount": 5,
                    "receiver": "Nikhil Kale",
                    "sender": "Aayush"
                },
                {
                    "amount": 15,
                    "receiver": "Aayush2",
                    "sender": "Nikhil"
                },
                {
                    "amount": 1,
                    "receiver": "Miner_5